# 03 · Vector Databases

> **Source notes:** `VectorDBs.md` + `VectorDBs_Supplement.md`

How does a vector store find the nearest neighbours in milliseconds across millions of vectors?

This notebook walks through:
- Distance metrics (L2, Cosine, Dot Product)
- Why brute-force search fails at scale
- IVF clustering-based ANN index
- HNSW graph-based ANN index (used by ChromaDB, Pinecone, Qdrant, Weaviate)
- DiskANN and quantization concepts
- Practical ChromaDB demo

**Tools (all local):** `numpy`, `scikit-learn`, `chromadb`, `sentence-transformers`

In [ ]:
## 0 · Setup
import subprocess, sys
pkgs = ["numpy", "scikit-learn", "chromadb", "sentence-transformers", "matplotlib"]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)
print("Packages ready.")

## 1 · Distance Metrics — How "Closeness" Is Measured

All vector search rests on a **distance (or similarity) function**:

| Metric | Formula | Use When |
|--------|---------|----------|
| **Euclidean (L2)** | `||a - b||` | Image embeddings, sensor data |
| **Cosine Similarity** | `(a · b) / (||a|| · ||b||)` | Text / semantic similarity |
| **Dot Product** | `a · b` | Recommendation; magnitude matters |

**Interview-critical fact:** when vectors are **L2-normalised** (unit length), cosine similarity and dot product give identical rankings, and maximising dot product equals minimising Euclidean distance.

Most embedding models (sentence-transformers with `normalize_embeddings=True`) output unit vectors, making the metric choice less critical in practice — but knowing *why* matters.

In [ ]:
import numpy as np

def l2(a, b):      return float(np.linalg.norm(a - b))
def cosine(a, b):  return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
def dot(a, b):     return float(np.dot(a, b))

# 4-dimensional toy example
q = np.array([1.0, 0.5, 0.2, 0.8])
a = np.array([0.9, 0.6, 0.1, 0.7])   # semantically close
b = np.array([0.1, 0.9, 0.8, 0.2])   # semantically distant

# Normalise to unit vectors
q_n = q / np.linalg.norm(q)
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)

print(f"{'Metric':<22} {'q vs a':>10} {'q vs b':>10}  {'Closer?':>8}")
print("-" * 55)
print(f"{'L2 (raw)':<22} {l2(q,a):>10.4f} {l2(q,b):>10.4f}  {'a' if l2(q,a)<l2(q,b) else 'b':>8}")
print(f"{'Cosine (raw)':<22} {cosine(q,a):>10.4f} {cosine(q,b):>10.4f}  {'a' if cosine(q,a)>cosine(q,b) else 'b':>8}")
print(f"{'Cosine (normalised)':<22} {cosine(q_n,a_n):>10.4f} {cosine(q_n,b_n):>10.4f}  {'a' if cosine(q_n,a_n)>cosine(q_n,b_n) else 'b':>8}")
print(f"{'Dot (normalised)':<22} {dot(q_n,a_n):>10.4f} {dot(q_n,b_n):>10.4f}  {'a' if dot(q_n,a_n)>dot(q_n,b_n) else 'b':>8}")
print()
print("Key: on normalised vectors, Cosine == Dot Product (same ranking).")

## 2 · Why Brute-Force Search Fails at Scale

**Exact search** = compute distance to **every** stored vector.

**Time complexity:** `O(N x d)` per query.

| Scale | Dimensions | Ops/query | Time at 10 GFLOP/s |
|-------|-----------|-----------|---------------------|
| 100 K | 384 | 38 M | ~4 ms OK |
| 10 M | 768 | 7.7 B | ~770 ms TOO SLOW |
| 100 M | 768 | 77 B | ~7.7 s WAY TOO SLOW |

**Memory:** 100 M vectors x 768 dims x 4 bytes (float32) = **307 GB** — exceeds most server RAM.

Traditional indexes (kd-trees, ball-trees) degrade in high dimensions — "the curse of dimensionality."

**Solution: Approximate Nearest Neighbour (ANN) indexes** — trade a tiny bit of recall for massive speed gains.

In [ ]:
import time

rng = np.random.default_rng(42)

def brute_force_search(corpus, query, k=5):
    """Exact cosine search (dot product on normalised vectors)."""
    sims = corpus @ query   # (N,)
    top_k = np.argpartition(sims, -k)[-k:]
    return top_k[np.argsort(sims[top_k])[::-1]]

print(f"{'N':>10}  {'dim':>5}  {'time (ms)':>12}")
print("-" * 35)
for n in [1_000, 10_000, 100_000]:
    corpus = rng.standard_normal((n, 384)).astype(np.float32)
    corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)
    query  = rng.standard_normal(384).astype(np.float32)
    query /= np.linalg.norm(query)

    start = time.perf_counter()
    _ = brute_force_search(corpus, query, k=5)
    elapsed_ms = (time.perf_counter() - start) * 1000

    print(f"{n:>10,}  {384:>5}  {elapsed_ms:>12.2f}")

## 3 · IVF — Inverted File Index

**Idea:** partition the vector space into **K clusters** (k-means). At query time only search the closest `nprobe` clusters instead of everything.

```
Training:
  K-means on corpus --> K centroids + N inverted lists

Query:
  1. Compute distance from query to all K centroids  (cheap: O(K * d))
  2. Pick nprobe closest centroids
  3. Search only the vectors in those clusters
```

**Key tunables:**
- `K` — number of clusters (more clusters = finer partitioning = faster, but lower recall per cluster)
- `nprobe` — clusters searched at query time (higher = better recall, slower)

In [ ]:
from sklearn.cluster import MiniBatchKMeans

N, D, K, nprobe = 20_000, 64, 50, 5

corpus = rng.standard_normal((N, D)).astype(np.float32)
corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)

# Build IVF index
print("Building IVF index (k-means) ...")
kmeans = MiniBatchKMeans(n_clusters=K, random_state=42, n_init="auto")
kmeans.fit(corpus)
cluster_ids = kmeans.labels_
inverted = {k: np.where(cluster_ids == k)[0] for k in range(K)}

query = rng.standard_normal(D).astype(np.float32)
query /= np.linalg.norm(query)

# Ground truth: brute force
t0 = time.perf_counter()
gt = brute_force_search(corpus, query, k=5)
bf_ms = (time.perf_counter() - t0) * 1000

# IVF search
t0 = time.perf_counter()
centroid_sims = kmeans.cluster_centers_ @ query
top_clusters = np.argsort(centroid_sims)[-nprobe:]
cand_idx = np.concatenate([inverted[c] for c in top_clusters])
cand_vecs = corpus[cand_idx]
loc_sims = cand_vecs @ query
top_local = np.argsort(loc_sims)[-5:][::-1]
ivf_result = cand_idx[top_local]
ivf_ms = (time.perf_counter() - t0) * 1000

recall = len(set(gt) & set(ivf_result)) / 5
print(f"Brute-force : {bf_ms:.2f} ms")
print(f"IVF (nprobe={nprobe}) : {ivf_ms:.2f} ms  |  Recall@5: {recall:.0%}")
print(f"Searched {len(cand_idx)}/{N} = {len(cand_idx)/N:.0%} of corpus")

## 4 · HNSW — Hierarchical Navigable Small World

HNSW is a **graph-based** ANN index used by most production vector DBs (ChromaDB, Pinecone, Qdrant, Weaviate, Azure AI Search).

```
Layer 2 (sparse long-range links)   A ----------- B
                                    |             |
Layer 1                             A --- C --- D-B
                                    |             |
Layer 0 (dense local links)         A-c-C-d-D-e-E-B  <- most vectors live here
```

**How search works:**
1. Enter at a random node in the **top (sparse) layer**
2. Greedily navigate toward the query (always step to the closer neighbour)
3. Drop to the layer below when stuck; repeat
4. At Layer 0, do a local exhaustive search among found neighbours

**Key parameters:**
- `M` — max edges per node (higher = better recall, more memory, slower build)
- `ef_construction` — size of dynamic list during build (higher = better quality graph)
- `ef` — size of candidate list at query time (higher = better recall, slower query)

ChromaDB uses HNSW internally with sensible defaults. We just configure the space (metric).

In [ ]:
# ChromaDB uses HNSW under the hood -- we just interact with the high-level API
import chromadb
from sentence_transformers import SentenceTransformer

em = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
col = client.get_or_create_collection(
    "vecdb-concepts",
    metadata={"hnsw:space": "cosine"},   # tell HNSW which distance to use
)

docs = [
    "IVF index uses k-means clustering to partition the vector space.",
    "HNSW is a graph-based approximate nearest neighbour index.",
    "DiskANN is designed for billion-scale search with SSD-backed storage.",
    "PQ (Product Quantization) compresses vectors to reduce memory.",
    "Flat brute-force search has perfect recall but scales poorly.",
    "nprobe controls how many IVF clusters are searched at query time.",
    "ef_construction determines HNSW build quality vs. build time tradeoff.",
    "Cosine similarity measures the angle between two vectors.",
    "FAISS is Facebook's open-source library for efficient vector search.",
]
vecs = em.encode(docs, normalize_embeddings=True).tolist()
col.add(ids=[f"d{i}" for i in range(len(docs))], embeddings=vecs, documents=docs)

# Query
q = "What index works best for billion-scale approximate search?"
q_vec = em.encode([q], normalize_embeddings=True).tolist()
res = col.query(query_embeddings=q_vec, n_results=3)

print(f"Query: '{q}'\n")
print("Top-3 results (via ChromaDB HNSW):")
for doc, dist in zip(res["documents"][0], res["distances"][0]):
    sim = 1 - dist   # ChromaDB returns cosine distance = 1 - similarity
    print(f"  [{sim:.3f}]  {doc}")

## 5 · DiskANN, Quantization & Index Selection

### DiskANN
- Stores the HNSW graph primarily on **SSD**, caching hot nodes in RAM
- Enables billion-scale search on commodity servers
- Used in Azure AI Search

### Product Quantization (PQ)
Compress vectors by splitting each high-dim vector into M sub-vectors, each quantised to a codebook:

| Original | PQ compressed | Compression |
|---------|--------------|-------------|
| 768 × 4 B = 3072 B | ~96 B | ~32x |

Trade-off: memory savings at the cost of some recall loss.

### Index Selection Guide

| Scenario | Recommended |
|----------|------------|
| < 100 K vectors, perfect recall | **Flat** (brute-force) |
| 100 K – 10 M, general purpose | **HNSW** |
| Large corpus, memory-constrained | **IVF + PQ** |
| Billions of vectors, commodity hardware | **DiskANN** |
| Streaming inserts (no rebuild) | **HNSW** (supports incremental adds) |

## 6 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| L2 / Cosine / Dot | On normalised vectors they give identical rankings |
| Brute-force | Perfect recall; O(N*d) cost; impractical above ~100 K |
| IVF | K-means partitioning; tune nprobe for recall vs. speed |
| HNSW | Graph-based greedy descent; dominant in production |
| DiskANN | HNSW for billion-scale; data lives on SSD |
| PQ | Compress vectors 32x; small recall loss |

**Next:** `04-react-agents/notebook.ipynb` — building an agent that uses all of this.

## 6 · PizzaBot Connection — Index Choice for a Small Corpus

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The PizzaBot menu corpus (~500 chunks) is small enough that the index choice is mostly invisible to users — but it makes the tradeoffs concrete:

| Scenario | Index | Why |
|---|---|---|
| **Prototype** | FAISS Flat (brute-force, exact) | 500 vectors; exact search < 1 ms; no tuning needed |
| **10k menu items** | FAISS `IndexHNSWFlat` | Sub-millisecond search; `efSearch=64` balances recall/latency |
| **Multi-store (1M+ items)** | IVF-PQ in a dedicated vector DB | Corpus distribution across stores matches IVF cluster assumptions |
| **Allergen safety queries** | Flat or `efSearch=128+` | Safety-critical: favour recall over latency — never miss a allergen flag |

**Key insight applied here:** cosine similarity is the right metric for MiniLM embeddings because they are not L2-normalised by default. The FAISS L2 metric would give different (incorrect) results.

**Try it:** add a FAISS cell that embeds the PizzaBot `allergens.csv` rows and queries `"dairy-free options"` — observe that it returns items without dairy even when the query phrasing doesn't match any row verbatim.
